# Feature Engineering
- Extrair, dos dados brutos, os melhores recursos (features) para o nosso modelo de forma a aumentar a acurácia
- Vamos entender quais são as melhores features após a análise exploratória
    - https://www.youtube.com/watch?v=4sxhE3wP3Ug&t=94s

In [5]:
import pandas as pd

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
baseLimpa = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/hastag/curso_youtube_lucas/07-11 - Feature Engineering em Python/ChavesClientes.xlsx")
baseLimpa.head()

,ID,ChaveSituacao,ClassRisco,CatCliente,Pagamento
0,1,32FC,Ccinza,Basic-Alpha,1
1,2,25MV,AAmarelo,Black,1
2,3,27MV,B-Amarelo,Basic-Beta,1
3,4,26FD,BAmarelo,Black,0
4,5,26FD,C-Amarelo,Black,0


In [8]:
baseLimpa = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/hastag/curso_youtube_lucas/07-11 - Feature Engineering em Python/ChavesClientesLimpo.xlsx")
baseLimpa.head()

,ChaveSituacao,ClassRisco,CatCliente,Pagamento,Idade,Genero,EstadoCivil,Categoria,CatVIP,Risco
0,32FC,Ccinza,Basic-Alpha,1,32,F,C,Basic,Alpha,C
1,25MV,AAmarelo,Black,1,25,M,V,Black,Comum,A
2,27MV,B-Amarelo,Basic-Beta,1,27,M,V,Basic,Beta,B-
3,26FD,BPreto,Black,0,26,F,D,Black,Comum,B
4,26FD,C-Amarelo,Black,0,26,F,D,Black,Comum,C-


**Podemos excluir as colunas que não vamos usar**

In [9]:
baseLimpa = baseLimpa.drop(['ChaveSituacao','ClassRisco','CatCliente'],axis=1)

**E verificar as informações da base**

In [10]:
baseLimpa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Pagamento    20 non-null     int64 
 1   Idade        20 non-null     int64 
 2   Genero       20 non-null     object
 3   EstadoCivil  20 non-null     object
 4   Categoria    20 non-null     object
 5   CatVIP       20 non-null     object
 6   Risco        20 non-null     object
dtypes: int64(2), object(5)
memory usage: 1.2+ KB


### Ao tentar colocar esses dados em um modelo como o de Regressão Linear vamos ter o seguinte erro

In [14]:
# Selecionando os valores de X e y
X = baseLimpa[['Idade','Genero','EstadoCivil','Categoria','CatVIP','Risco']]
y = baseLimpa.Pagamento

from sklearn.linear_model import LinearRegression
reg = LinearRegression().fit(X, y)

reg.score(X,y)

ValueError: ignored

### Por isso precisamos conseguir tratar os dados antes de inserir no modelo

In [15]:
baseLimpa.head(2)

,Pagamento,Idade,Genero,EstadoCivil,Categoria,CatVIP,Risco
0,1,32,F,C,Basic,Alpha,C
1,1,25,M,V,Black,Comum,A


**Com o One Hot Encoding podemos tratar valores que não tem relação de ordem entre eles**
- https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html

In [16]:
# Importando e utilizando o OneHotEncoder para as colunas 'Genero' e 'EstadoCivil'
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder()
ohe_transform = ohe.fit_transform(baseLimpa[['Genero','EstadoCivil']])


In [18]:
# Nome das features
ohe.get_feature_names_out()

array(['Genero_F', 'Genero_M', 'EstadoCivil_C', 'EstadoCivil_D',
       'EstadoCivil_S', 'EstadoCivil_V'], dtype=object)

In [19]:
# Array de valores
ohe_transform.toarray()

array([[1., 0., 1., 0., 0., 0.],
       [0., 1., 0., 0., 0., 1.],
       [0., 1., 0., 0., 0., 1.],
       [1., 0., 0., 1., 0., 0.],
       [1., 0., 0., 1., 0., 0.],
       [1., 0., 1., 0., 0., 0.],
       [0., 1., 0., 1., 0., 0.],
       [0., 1., 0., 1., 0., 0.],
       [1., 0., 0., 0., 1., 0.],
       [0., 1., 0., 0., 0., 1.],
       [0., 1., 0., 0., 0., 1.],
       [1., 0., 1., 0., 0., 0.],
       [1., 0., 1., 0., 0., 0.],
       [0., 1., 1., 0., 0., 0.],
       [0., 1., 1., 0., 0., 0.],
       [0., 1., 1., 0., 0., 0.],
       [1., 0., 0., 1., 0., 0.],
       [0., 1., 0., 1., 0., 0.],
       [0., 1., 0., 1., 0., 0.],
       [1., 0., 0., 0., 0., 1.]])

In [23]:
# Transformando esses dados em um DataFrame
df_ohe = pd.DataFrame(ohe_transform.toarray())
df_ohe.columns = ohe.get_feature_names_out()
df_ohe.head()

,Genero_F,Genero_M,EstadoCivil_C,EstadoCivil_D,EstadoCivil_S,EstadoCivil_V
0,1.0,0.0,1.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0,1.0
2,0.0,1.0,0.0,0.0,0.0,1.0
3,1.0,0.0,0.0,1.0,0.0,0.0
4,1.0,0.0,0.0,1.0,0.0,0.0


In [28]:
# Para finalizar, podemos concatenar as duas bases
baseLimpa = pd.concat([baseLimpa,df_ohe], axis=1)
baseLimpa.head(2)

,Pagamento,Idade,Genero,EstadoCivil,Categoria,CatVIP,Risco,Genero_F,Genero_M,EstadoCivil_C,EstadoCivil_D,EstadoCivil_S,EstadoCivil_V,Genero_F,Genero_M,EstadoCivil_C,EstadoCivil_D,EstadoCivil_S,EstadoCivil_V
0,1,32,F,C,Basic,Alpha,C,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1,1,25,M,V,Black,Comum,A,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0


**Já se os valores tiverem uma relação de ordem, podemos usar o Ordinal Encoding**
- https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html#sklearn.preprocessing.OrdinalEncoder

In [26]:
# Entendendo a relação entre a coluna "Categoria"
baseLimpa.Categoria.value_counts()

Black       7
Platinum    7
Basic       6
Name: Categoria, dtype: int64

In [30]:
# Importando e utilizando o OrdinalEncoder para a coluna 'Categoria'
from sklearn.preprocessing import OrdinalEncoder
oe = OrdinalEncoder()
oe_transform = oe.fit_transform(baseLimpa.Categoria.values.reshape(-1, 1))

In [31]:
# E podemos adicionar essa coluna
baseLimpa['NrCategoria'] = oe_transform

In [32]:
# Visualizando a base
baseLimpa.head(5)

,Pagamento,Idade,Genero,EstadoCivil,Categoria,CatVIP,Risco,Genero_F,Genero_M,EstadoCivil_C,EstadoCivil_D,EstadoCivil_S,EstadoCivil_V,Genero_F,Genero_M,EstadoCivil_C,EstadoCivil_D,EstadoCivil_S,EstadoCivil_V,NrCategoria
0,1,32,F,C,Basic,Alpha,C,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
1,1,25,M,V,Black,Comum,A,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
2,1,27,M,V,Basic,Beta,B-,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,0,26,F,D,Black,Comum,B,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
4,0,26,F,D,Black,Comum,C-,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0


In [36]:
# Fazendo o mesmo para a coluna risco
oe = OrdinalEncoder(categories=[['C-','C','C+','B-','B','B+','A-','A','A+']])
oe_transform_risco = oe.fit_transform(baseLimpa.Risco.values.reshape(-1, 1))

In [37]:
baseLimpa['NrRisco'] = oe_transform_risco

In [38]:
# Visualizando a base
baseLimpa.head(5)

,Pagamento,Idade,Genero,EstadoCivil,Categoria,CatVIP,Risco,Genero_F,Genero_M,EstadoCivil_C,...,EstadoCivil_S,EstadoCivil_V,Genero_F,Genero_M,EstadoCivil_C,EstadoCivil_D,EstadoCivil_S,EstadoCivil_V,NrCategoria,NrRisco
0,1,32,F,C,Basic,Alpha,C,1.0,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,1,25,M,V,Black,Comum,A,0.0,1.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,7.0
2,1,27,M,V,Basic,Beta,B-,0.0,1.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,3.0
3,0,26,F,D,Black,Comum,B,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,4.0
4,0,26,F,D,Black,Comum,C-,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


**Por fim, podemos criar funções para transformar colunas como transformar a CatVIP para verificar apenas se o cliente é VIP ou não**

In [43]:
# Criando uma função para verificar se o cliente é VIP
def define_VIP(valor):
  if valor == 'Alpha' or valor == 'Beta':
    return 1
  else:
    return 0

In [44]:
# Aplicando essa função na coluna 'CatVIP'
baseLimpa["NrVip"] = baseLimpa.CatVIP.apply(define_VIP)

In [45]:
# Visualizando a base
baseLimpa.head(5)

,Pagamento,Idade,Genero,EstadoCivil,Categoria,CatVIP,Risco,Genero_F,Genero_M,EstadoCivil_C,...,EstadoCivil_V,Genero_F,Genero_M,EstadoCivil_C,EstadoCivil_D,EstadoCivil_S,EstadoCivil_V,NrCategoria,NrRisco,NrVip
0,1,32,F,C,Basic,Alpha,C,1.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1
1,1,25,M,V,Black,Comum,A,0.0,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,7.0,0
2,1,27,M,V,Basic,Beta,B-,0.0,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,3.0,1
3,0,26,F,D,Black,Comum,B,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,4.0,0
4,0,26,F,D,Black,Comum,C-,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0


**Limpando novamente as colunas desnecessárias**

In [46]:
# Retirando novamente as colunas desnecessárias
baseLimpa = baseLimpa.drop(['Genero','EstadoCivil','Categoria','CatVIP','Risco'], axis=1)
baseLimpa.head(5)

,Pagamento,Idade,Genero_F,Genero_M,EstadoCivil_C,EstadoCivil_D,EstadoCivil_S,EstadoCivil_V,Genero_F,Genero_M,EstadoCivil_C,EstadoCivil_D,EstadoCivil_S,EstadoCivil_V,NrCategoria,NrRisco,NrVip
0,1,32,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1
1,1,25,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,7.0,0
2,1,27,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,3.0,1
3,0,26,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,4.0,0
4,0,26,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0


### Usando novamente em um modelo de Regressão Linear

In [47]:
# Selecionando os valores de X e y
X = baseLimpa.drop('Pagamento',axis=1)
y = baseLimpa.Pagamento

from sklearn.linear_model import LinearRegression
reg = LinearRegression().fit(X, y)

reg.score(X,y)

0.6197521275369209